# 🌲 Python DP on Trees — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> DP on Trees is like asking every branch manager to report the best result from their team
> before making a decision at the top. Each node in the tree is a manager.
> They gather info from their left and right sub-managers (children), compute the best
> local result, and pass a summary upward. The root gets everyone's best and makes the final call.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is DP on Trees? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Patterns](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: House Robber III (LC 337)](#5) |
| 6 | [Pattern 2: Diameter of Binary Tree (LC 543)](#6) |
| 7 | [Pattern 3: Binary Tree Maximum Path Sum (LC 124)](#7) |
| 8 | [The DP on Trees Decision Map](#8) |
| 9 | [Interview Cheat Sheet](#9) |

<a id='1'></a>
## 1. What Is DP on Trees? The Visual Model

```
               DP ON TREES — THE BRANCH MANAGER REPORT

  Tree:
           3
          / \
         2   3
          \   \
           3   1

  House Robber III — rob or skip each node:

  Each node returns (rob_this, skip_this):
    rob_this  = node.val + skip_left + skip_right  (rob here, skip children)
    skip_this = max(rob_left, skip_left) + max(rob_right, skip_right)  (skip here, best children)

  Leaf 3:   (rob=3, skip=0)
  Leaf 1:   (rob=1, skip=0)
  Node 2:   rob=2+0=2, skip=max(3,0)=3  → (2, 3)
  Node 3:   rob=3+0=3, skip=max(1,0)=1  → (3, 1)
  Root 3:   rob=3+3+1=7, skip=max(2,3)+max(3,1)=3+3=6  → (7, 6)
  Answer = max(7, 6) = 7

  KEY PATTERN:
  DFS post-order: process children BEFORE the current node.
  Return enough info from each subtree for the parent to make an optimal decision.
  Track global answer in a variable (or list for closure mutation).
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
from collections import deque

# TreeNode — standard LeetCode definition
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

# make_tree — BFS builder from level-order list
def make_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue = deque([root])
    i = 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

# DP on Trees: post-order DFS template
def post_order_dp(root):
    if not root:
        return 0     # base case: empty subtree contributes 0
    left  = post_order_dp(root.left)    # gather left subtree result
    right = post_order_dp(root.right)   # gather right subtree result
    # compute answer for current node using left + right results
    return left + right + root.val       # example: subtree sum

tree = make_tree([3,2,3,None,3,None,1])
print("subtree sum:", post_order_dp(tree))  # 3+2+3+3+1=12
print("Tree helpers loaded.")

<a id='3'></a>
## 3. The Core API — All Patterns

```
DP ON TREES PATTERN         WHAT DFS RETURNS          PROBLEM TYPE
────────────────────────────────────────────────────────────────────────
Rob/skip node               (rob_here, skip_here)     House Robber III
Max path through node       single int (arm length)   Diameter, Path Sum
Check property subtree      bool                      Balanced tree
Accumulate with global max  arm length + global update Max path sum, Diameter

POST-ORDER TEMPLATE:
  def dp(node):
      if not node: return BASE_CASE
      left  = dp(node.left)   # left subtree result
      right = dp(node.right)  # right subtree result
      # update global answer if path passes through this node
      ans = max(ans, left + right + node.val)   # example
      return local_value_for_parent              # what parent needs

THINGS YOU DO NOT DO:
❌  Process children AFTER the current node — must be post-order (children first)
❌  Returning the full path value to the parent (path through node can't be extended both ways)
❌  Using a module-level variable for global state — use a list [0] for closure mutation
❌  Forgetting base case for None nodes — always handle the leaf's leaf
```

In [ ]:
# DEMO: two ways to pass global state in tree DP

# Method 1: list as mutable closure (Pythonic, avoids nonlocal)
def tree_max_val_list(root):
    global_max = [float('-inf')]   # list so inner function can mutate it
    def dfs(node):
        if not node: return 0
        left  = dfs(node.left)
        right = dfs(node.right)
        global_max[0] = max(global_max[0], node.val)  # update global
        return node.val  # return to parent
    dfs(root)
    return global_max[0]

# Method 2: nonlocal keyword
def tree_max_val_nonlocal(root):
    global_max = float('-inf')
    def dfs(node):
        nonlocal global_max
        if not node: return 0
        left  = dfs(node.left)
        right = dfs(node.right)
        global_max = max(global_max, node.val)
        return node.val
    dfs(root)
    return global_max

tree = make_tree([3, 2, 3, None, 3, None, 1])
print("max val (list method):    ", tree_max_val_list(tree))     # 3
print("max val (nonlocal method):", tree_max_val_nonlocal(tree)) # 3
print("Global state patterns demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                    WHAT TO DO
──────────────────────────────────────────────────────────────────────
"rob houses in a tree"                   post-order, return (rob,skip) pair
"diameter of tree" (longest path)        post-order, height+height, update global
"max path sum" (path can go up+down)     post-order, left+right arm, update global
"balanced tree"                          post-order, return -1 if unbalanced
"max/min value in subtree"              post-order, return up; update global
"count nodes with property"              post-order accumulate count
"deepest common ancestor"               post-order propagate match flags
```

<a id='5'></a>
## 5. 🧩 Pattern 1: House Robber III — LC 337

---

```
PROBLEM:
  Rob houses arranged in a binary tree. Adjacent nodes (parent-child) have an alarm.
  Maximize the total stolen without robbing directly linked nodes.

TRICK:
  Post-order DFS. Each node returns a pair:
    rob  = max money if we ROB  this node (skip both children)
    skip = max money if we SKIP this node (take best from each child independently)

  rob  = node.val + left.skip + right.skip
  skip = max(left.rob, left.skip) + max(right.rob, right.skip)

  Answer at root = max(root.rob, root.skip)

SLOW MOTION TRACE on [3,2,3,None,3,None,1]:

  Tree:      3
            / \
           2   3
            \   \
             3   1

  Leaf 3 (left child of 2):  rob=3, skip=0 → (3,0)
  Leaf 1 (right child of 3): rob=1, skip=0 → (1,0)
  Node 2: rob=2+0+0=2, skip=max(3,0)+max(0,0)=3+0=3 → (2,3)
  Node 3: rob=3+0+0=3, skip=max(0,0)+max(1,0)=0+1=1 → (3,1)
  Root 3: rob=3+3+1=7, skip=max(2,3)+max(3,1)=3+3=6 → (7,6)
  Answer = max(7,6) = 7

KEY INSIGHT:
  Return BOTH options from each node. Let the parent decide which combination is optimal.
  This avoids an exponential brute-force — each state computed exactly once.

TIME:  O(n) — visit each node once
SPACE: O(h) — recursion stack
```

In [ ]:
def rob_tree(root):
    """
    LC 337 — House Robber III
    Approach: Post-order DFS returning (rob_here, skip_here) pair for each node.
    Args:
        root (TreeNode): root of the house tree.
    Returns:
        int: maximum money that can be robbed.
    Time:  O(n) — each node visited exactly once
    Space: O(h) — recursion stack depth = tree height
    """
    def dfs(node):
        if not node:
            return (0, 0)   # (rob, skip) for empty subtree

        left_rob,  left_skip  = dfs(node.left)   # left branch manager's report
        right_rob, right_skip = dfs(node.right)  # right branch manager's report

        rob  = node.val + left_skip + right_skip   # rob this node → must skip children
        skip = max(left_rob, left_skip) + max(right_rob, right_skip)  # skip this → best children

        return (rob, skip)

    rob, skip = dfs(root)
    return max(rob, skip)

# Slow motion on [3,2,3,None,3,None,1]:
# leaf 3: (3,0), leaf 1: (1,0)
# node 2: rob=2+0+0=2, skip=max(3,0)=3 → (2,3)
# node 3(right): rob=3+0+0=3, skip=max(1,0)=1 → (3,1)
# root 3: rob=3+3+1=7, skip=max(2,3)+max(3,1)=3+3=6 → (7,6)
# return max(7,6)=7

def test_harness(fn):
    tests = [
        ([3,2,3,None,3,None,1], 7),
        ([3,4,5,1,3,None,1], 9),
        ([1], 1),
        ([4,1,None,2,None,3], 7),  # rob 4 + rob 3
        ([2,1,3,None,4], 7),
    ]
    passed = 0
    for *inputs, expected in tests:
        root = make_tree(inputs[0])
        got = fn(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(rob_tree)
print("rob_tree defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Diameter of Binary Tree — LC 543

---

```
PROBLEM:
  Return the length (number of edges) of the longest path between any two nodes
  in a binary tree. The path may not pass through the root.

TRICK:
  At each node, the longest path THROUGH that node = height(left) + height(right).
  Post-order DFS returns the height of each subtree.
  Maintain a global maximum for the diameter (may be found deep in the tree).

SLOW MOTION TRACE on [1,2,3,4,5]:

  Tree:     1
           / \
          2   3
         / \
        4   5

  Leaves 4, 5: height=0 each
  Node 2: left_h=0+1=1, right_h=0+1=1; path_thru=1+1=2; return height=1+max(1,1)=2
  Node 3: leaf; height=0
  Root 1: left_h=2, right_h=0; path_thru=2+0=2 (no improvement); return 3

  Wait — diameter at node 2 = 4→2→5 or 4→2→1→3 etc.
  path through node 2 = left_height(1) + right_height(1) = 1+1 = 2 edges → but actually:
  dfs returns # of NODES in path from node to deepest leaf.
  diameter = left_arm + right_arm (# of edges through this node)

  Leaf 4: return 1 (1 node: just itself)
  Leaf 5: return 1
  Node 2: left=1,right=1; diameter_thru=1+1=2 edges; return 1+max(1,1)=2
  Leaf 3: return 1
  Root 1: left=2,right=1; diameter_thru=2+1=3 edges ← global best
  answer = 3 (path 4→2→1→3 or 5→2→1→3)

KEY INSIGHT:
  DFS returns node count (height). Diameter = left_arm + right_arm at each node.
  Global max updated at every node — diameter may not pass through root.

TIME:  O(n)
SPACE: O(h)
```

In [ ]:
def diameter_of_binary_tree(root):
    """
    LC 543 — Diameter of Binary Tree
    Approach: Post-order DFS returns arm length (node count); diameter = left+right arms.
    Args:
        root (TreeNode): root of the binary tree.
    Returns:
        int: length (edges) of the longest path between any two nodes.
    Time:  O(n) — each node visited once
    Space: O(h) — recursion stack
    """
    diameter = [0]   # mutable container for global maximum

    def dfs(node):
        if not node:
            return 0   # empty subtree contributes 0 nodes
        left_arm  = dfs(node.left)   # longest arm going left (in node count)
        right_arm = dfs(node.right)  # longest arm going right
        # path through this node spans left_arm + right_arm edges
        diameter[0] = max(diameter[0], left_arm + right_arm)
        return 1 + max(left_arm, right_arm)  # return arm length to parent (+1 for this node)

    dfs(root)
    return diameter[0]

# Slow motion on [1,2,3,4,5]:
# dfs(4)=1, dfs(5)=1
# dfs(2): left=1,right=1; diameter=max(0,2)=2; return 1+max(1,1)=2
# dfs(3): return 1
# dfs(1): left=2,right=1; diameter=max(2,3)=3; return 1+max(2,1)=3
# return 3

def test_harness(fn):
    tests = [
        ([1,2,3,4,5], 3),
        ([1,2], 1),
        ([1], 0),
        ([4,-7,-3,None,None,-9,-3,9,-7,-4,None,6,None,-6,-6,None,None,0,6,5,None,9,None,None,-1,-4,None,None,None,-2], 8),
        ([1,2,3,None,4,None,5], 5),   # zigzag longest
    ]
    passed = 0
    for *inputs, expected in tests:
        root = make_tree(inputs[0])
        got = fn(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(diameter_of_binary_tree)
print("diameter_of_binary_tree defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Binary Tree Maximum Path Sum — LC 124

---

```
PROBLEM:
  Given a binary tree with possibly negative values, find the maximum path sum.
  A path is any sequence of nodes along edges — does not need to pass through root.

TRICK:
  Same structure as Diameter, but with values.
  At each node:
    global_max = max(global_max, left_arm + right_arm + node.val)
    return node.val + max(left_arm, right_arm, 0)  ← can discard negative arms

  Key difference vs Diameter:
  - Values can be negative → always clamp arms to max(arm, 0)
    (a negative arm makes the path worse — better to not extend in that direction)
  - Path through node includes node.val, not just edge count

SLOW MOTION TRACE on [-10,9,20,None,None,15,7]:

  Tree:     -10
            /  \
           9   20
               / \
              15   7

  dfs(9):  no children; global_max=max(-inf,9)=9; return 9
  dfs(15): no children; global_max=max(9,15)=15; return 15
  dfs(7):  no children; global_max=max(15,7)=15; return 7
  dfs(20): left=15, right=7; path_thru=15+7+20=42; global_max=42; return 20+max(15,7)=35
  dfs(-10): left=9, right=35;
    path_thru=9+35+(-10)=34; global_max=max(42,34)=42
    return -10+max(9,35)=25 (to root's parent, irrelevant)
  answer = 42 (path: 15→20→7)

KEY INSIGHT:
  Return value to parent = best single arm extension (one direction only).
  Global max update uses BOTH arms (full path through this node).
  Clamp arms to 0: max(arm, 0) prevents negative contributions.

TIME:  O(n)
SPACE: O(h)
```

In [ ]:
def max_path_sum(root):
    """
    LC 124 — Binary Tree Maximum Path Sum
    Approach: Post-order DFS; update global with both arms; return best single arm.
    Args:
        root (TreeNode): root of binary tree (values may be negative).
    Returns:
        int: maximum sum of any path in the tree.
    Time:  O(n) — each node visited once
    Space: O(h) — recursion stack
    """
    global_max = [float('-inf')]   # start at -inf: tree may have all-negative values

    def dfs(node):
        if not node:
            return 0   # empty branch contributes nothing

        left_arm  = max(dfs(node.left),  0)  # clamp to 0: discard negative arms
        right_arm = max(dfs(node.right), 0)  # a negative arm hurts the path sum

        # full path through this node (can't extend BOTH arms to parent)
        global_max[0] = max(global_max[0], left_arm + right_arm + node.val)

        # return the best single-direction arm to the parent (can only extend one way)
        return node.val + max(left_arm, right_arm)

    dfs(root)
    return global_max[0]

# Slow motion on [-10,9,20,None,None,15,7]:
# dfs(9)=9; dfs(15)=15; dfs(7)=7
# dfs(20): left=max(15,0)=15, right=max(7,0)=7
#   global=max(-inf,15+7+20)=42; return 20+max(15,7)=35
# dfs(-10): left=max(9,0)=9, right=max(35,0)=35
#   global=max(42,9+35-10)=42; return -10+max(9,35)=25
# return 42

def test_harness(fn):
    tests = [
        ([1,2,3], 6),
        ([-10,9,20,None,None,15,7], 42),
        ([-3], -3),          # all negative — must include at least one node
        ([1,-2,3], 4),       # 1 + 3 = 4 (skip -2)
        ([2,-1], 2),
        ([-1,-2,-3], -1),    # best single node
    ]
    passed = 0
    for *inputs, expected in tests:
        root = make_tree(inputs[0])
        got = fn(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(max_path_sum)
print("max_path_sum defined.")

<a id='8'></a>
## 8. The DP on Trees Decision Map

```
QUESTION TYPE                         RETURNS FROM DFS         LC
────────────────────────────────────────────────────────────────────
House robber on tree                  (rob, skip) pair          337
Diameter (longest path by edges)      arm length (node count)   543
Max path sum (values on nodes)        arm value (clamped≥0)     124
Balanced tree                         height or -1 sentinel     110
Count good nodes                      count propagated up        1448
Max depth                             depth (int)                104
Distribute coins                      excess propagated up       979

DFS RETURN VALUE RULES:
  What parent needs ≠ what we want for global answer.
  Return: what helps the PARENT make its decision.
  Update: global max/result at each node using both children's info.

  Diameter: return arm_length (single direction) — parent computes left+right
  Max path: return node.val + max(arm,0) — clamp negative, one direction only
  Rob/Skip: return (rob, skip) tuple — parent picks best combination
```

<a id='9'></a>
## 9. Interview Cheat Sheet

**1. When to reach for DP on Trees:**

| Signal | What to Do |
|--------|------------|
| "rob adjacent nodes in a tree" | post-order, return (rob, skip) |
| "longest path / diameter" | post-order, arm length + global update |
| "max sum path" | post-order, clamp arms ≥0, global update |
| "check balanced / valid property" | post-order, return -1 as sentinel |

**2. The post-order DFS template — memorize this:**

```python
def dfs(node):
    if not node: return BASE_CASE
    left  = dfs(node.left)    # process children FIRST (post-order)
    right = dfs(node.right)
    # update global answer using both children
    global_ans[0] = max(global_ans[0], combine(left, right, node.val))
    # return what the PARENT needs (single-direction info)
    return local_value_for_parent
```

**3. Common templates:**

```python
# HOUSE ROBBER III — return tuple
def dfs(node):
    if not node: return (0, 0)
    lr, ls = dfs(node.left)
    rr, rs = dfs(node.right)
    rob  = node.val + ls + rs
    skip = max(lr,ls) + max(rr,rs)
    return (rob, skip)

# DIAMETER — arm length
def dfs(node):
    if not node: return 0
    l, r = dfs(node.left), dfs(node.right)
    ans[0] = max(ans[0], l + r)     # update with both arms
    return 1 + max(l, r)            # return single arm to parent

# MAX PATH SUM — clamp arms
def dfs(node):
    if not node: return 0
    l = max(dfs(node.left),  0)     # clamp: discard negative arms
    r = max(dfs(node.right), 0)
    ans[0] = max(ans[0], l + r + node.val)
    return node.val + max(l, r)     # single direction to parent
```

**4. Gotchas to not forget:**

```
❌  Returning BOTH arms to parent — parent can only extend path in ONE direction
❌  Not clamping arms to 0 in max path sum — negative subtrees hurt the result
❌  Initializing global_max to 0 — tree may be all-negative (use float('-inf'))
❌  Using a module-level global — use a list [val] or nonlocal for closure
✅  Post-order: children always processed before the current node
✅  Rob/Skip tuple: return what the PARENT needs to combine optimally
✅  Diameter: count edges (not nodes) — the arm length IS the node count
✅  Max path: update global with left+right+val; return val+max(l,r) to parent
```

## Summary Map

```
                    🌲 DP ON TREES
                          │
              POST-ORDER DFS (children first)
                          │
           ┌──────────────┼──────────────┐
           │              │              │
      ROB/SKIP        ARM LENGTH     PATH SUM
      TUPLE           + GLOBAL       + GLOBAL
           │              │              │
      return           return         return
      (rob,skip)       1+max(l,r)     val+max(l,r)
      parent picks     parent adds    clamp arms≥0
      max(rob,skip)    left+right     left+right+val
      LC 337           LC 543         LC 124

CORE RULE:
  Post-order: gather from children, update global, return to parent.
  Return value = what parent needs (single arm).
  Global update = uses BOTH arms (full path through this node).
  Clamp negative arms to 0 when working with path sums.
```

---
*End of DP on Trees Master Guide — Sean Edition*